### Упражнения по разделу 1

**Упражнение 1. `flight_label`**

Напишите скалярную функцию `flight_label(p_flight_no VARCHAR, p_airport VARCHAR) RETURNS TEXT`, возвращающую строку вида `'SU1234 (DME)'`. Пометьте корректной категорией волатильности. Выведите метки всех рейсов:

```sql
SELECT flight_label(flight_no, departure_airport) FROM flights;
```

Ожидаемый вывод: `SU1234 (DME)`, `SU5678 (DME)`, `AB999 (SVO)`.

**Упражнение 2. `tickets_for_flight`**

Напишите табличную функцию `tickets_for_flight(p_flight_id INT) RETURNS TABLE(ticket_no VARCHAR, amount NUMERIC)`, возвращающую все билеты на указанный рейс, отсортированные по `amount` по убыванию. Категория — `STABLE`. Проверьте вызов `SELECT * FROM tickets_for_flight(1);` — должны увидеть два билета.

In [32]:
# Упражнение 1. flight_label

# Что делаем: функция берёт номер рейса и аэропорт и склеивает их в строку вида SU1234 (DME).
# Результат зависит только от аргументов, поэтому категория IMMUTABLE.

q("""
CREATE OR REPLACE FUNCTION flight_label(p_flight_no VARCHAR, p_airport VARCHAR)
RETURNS TEXT AS $$
BEGIN
    -- склеиваем номер рейса и аэропорт в скобках
    RETURN p_flight_no || ' (' || p_airport || ')';
END;
$$ LANGUAGE plpgsql IMMUTABLE   -- результат зависит только от аргументов
""")

# проверка: должно быть SU1234 (DME), SU5678 (DME), AB999 (SVO)
for row in q("SELECT flight_label(flight_no, departure_airport) FROM flights"):
    print(row[0])

SU1234 (DME)
SU5678 (DME)
AB999 (SVO)


In [33]:
# Упражнение 2. tickets_for_flight

# Что делаем: функция возвращает таблицу со всеми билетами одного рейса, самые дорогие сверху.
# Она только читает данные, поэтому STABLE.

q("""
CREATE OR REPLACE FUNCTION tickets_for_flight(p_flight_id INT)
RETURNS TABLE(ticket_no VARCHAR, amount NUMERIC) AS $$
BEGIN
    -- берём билеты нужного рейса, сортируем по цене от большей к меньшей
    -- (пишем t.ticket_no и t.amount с алиасом t, чтобы не путать
    --  колонки таблицы с колонками результата функции)
    RETURN QUERY
        SELECT t.ticket_no, t.amount
        FROM ticket_flights t
        WHERE t.flight_id = p_flight_id
        ORDER BY t.amount DESC;
END;
$$ LANGUAGE plpgsql STABLE      -- только читает таблицы
""")

# проверка: для рейса 1 должно быть два билета (18000.50 и 15000.00)
for row in q("SELECT * FROM tickets_for_flight(1)"):
    print(row)

('1234567891', Decimal('18000.50'))
('1234567890', Decimal('15000.00'))


### Упражнение по разделу 2

**Упражнение 3. Аудит для `equipment`**

Создайте ROW-триггер аудита для таблицы `equipment` (INSERT/UPDATE/DELETE → запись в `audit_log`). Используйте уже созданную функцию `log_application_changes_row()` — она универсальна и берёт имя таблицы из `TG_TABLE_NAME`.

Затем:
1. Вставьте две записи в `equipment`.
2. Обновите одну из них.
3. Удалите одну.
4. Выведите `operation` из `audit_log` для `table_name = 'equipment'`. Ожидаемая последовательность: `I`, `I`, `U`, `D`.

In [36]:
# Упражнение 3. Аудит для equipment

# Что делаем: вешаем на equipment готовую функцию аудита log_application_changes_row(). 
# После этого каждое изменение таблицы будет записываться в audit_log. 
# Затем делаем 2 вставки, 1 обновление и 1 удаление и смотрим, что записалось.

# подключаем уже готовую функцию аудита к таблице equipment
q("DROP TRIGGER IF EXISTS trg_audit_equipment_row ON equipment")
q("""
CREATE TRIGGER trg_audit_equipment_row
AFTER INSERT OR UPDATE OR DELETE ON equipment
FOR EACH ROW EXECUTE FUNCTION log_application_changes_row()
""")

# 1. две вставки
q("INSERT INTO equipment (name) VALUES ('Станок 1'), ('Станок 2')")

# 2. обновляем одну запись
q("UPDATE equipment SET name = 'Станок 1 (новый)' WHERE name = 'Станок 1'")

# 3. удаляем другую
q("DELETE FROM equipment WHERE name = 'Станок 2'")

# 4. смотрим журнал: должно быть I, I, U, D
for row in q("SELECT operation FROM audit_log WHERE table_name = 'equipment' ORDER BY log_id"):
    print(row[0])

# уборка, чтобы дальше (демо 3.1 использует id=1) всё работало как в семинаре
q("TRUNCATE equipment RESTART IDENTITY")

I
I
U
D


### Упражнение по разделу 3

**Упражнение 4. `updated_at` для `loan_applications`**

Создайте `BEFORE UPDATE`-триггер на таблицу `loan_applications`, который:
1. Автоматически проставляет `NEW.updated_at := now()`.
2. Запрещает через `RAISE EXCEPTION` изменение `status` у заявок со статусом `'APPROVED'`.

Проверка:
1. Вставьте заявку со `status = 'APPROVED'`.
2. Попробуйте `UPDATE loan_applications SET status = 'REJECTED' WHERE id = ...` — должно упасть.
3. `UPDATE loan_applications SET amount = 999 WHERE id = ...` — должно пройти, и `updated_at` заполниться.

In [38]:
# Упражнение 4. Триггер BEFORE UPDATE для loan_applications

# Что делаем: до сохранения изменения триггер делает две вещи:

# Всегда ставит в updated_at текущее время.
# Если заявка уже APPROVED и кто-то пытается сменить статус, он выбрасывает ошибку.

# Менять другие поля (например, amount) при этом можно.

q("""
CREATE OR REPLACE FUNCTION loan_applications_before()
RETURNS TRIGGER AS $$
BEGIN
    -- запрет: у одобренной заявки статус менять нельзя
    IF OLD.status = 'APPROVED' AND NEW.status <> OLD.status THEN
        RAISE EXCEPTION 'Нельзя менять статус одобренной заявки (id=%)', OLD.id;
    END IF;

    -- автозаполнение: ставим время последнего изменения
    NEW.updated_at := now();

    RETURN NEW;   -- пропускаем изменение дальше
END;
$$ LANGUAGE plpgsql
""")

q("DROP TRIGGER IF EXISTS trg_loan_before ON loan_applications")
q("""
CREATE TRIGGER trg_loan_before
BEFORE UPDATE ON loan_applications
FOR EACH ROW EXECUTE FUNCTION loan_applications_before()
""")

# 1. вставляем одобренную заявку и запоминаем её id
loan_id = q("""
    INSERT INTO loan_applications (borrower_name, amount, status)
    VALUES ('Дмитрий', 300000, 'APPROVED') RETURNING id
""")[0][0]

# 2. пробуем сменить статус: должна быть ошибка
try:
    q("UPDATE loan_applications SET status = 'REJECTED' WHERE id = %s", (loan_id,))
except psycopg.errors.RaiseException as e:
    print("Статус менять нельзя →", str(e).splitlines()[0])

# 3. меняем сумму: должно пройти, и updated_at заполнится
q("UPDATE loan_applications SET amount = 999 WHERE id = %s", (loan_id,))
print(q("SELECT id, amount, status, updated_at FROM loan_applications WHERE id = %s", (loan_id,)))

Статус менять нельзя → Нельзя менять статус одобренной заявки (id=5)
[(5, Decimal('999'), 'APPROVED', datetime.datetime(2026, 11, 20, 9, 17, 4, 433026))]


### Упражнения по разделу 4

**Упражнение 5. Обновляемое VIEW с `WITH CHECK OPTION`**

Создайте VIEW `v_svo_flights` с полями `flight_id, flight_no, departure_airport, scheduled_departure` и фильтром `departure_airport = 'SVO'`, добавьте `WITH CHECK OPTION`.

Проверьте:
1. `INSERT INTO v_svo_flights (...) VALUES ('XX111', 'SVO', ...)` — работает.
2. `INSERT INTO v_svo_flights (...) VALUES ('XX222', 'DME', ...)` — отклоняется.
3. `SELECT count(*) FROM v_svo_flights` — отражает только SVO-рейсы.

**Упражнение 6. MV по аэропортам + `REFRESH CONCURRENTLY`**

Создайте MATERIALIZED VIEW `mv_airport_stats`, агрегирующий количество рейсов по `departure_airport`:

```sql
SELECT departure_airport, count(*) AS flights_cnt
FROM flights
GROUP BY departure_airport;
```

1. Создайте уникальный индекс по `departure_airport`.
2. Выведите содержимое MV.
3. Добавьте новый рейс в `flights`.
4. Убедитесь, что MV не изменился.
5. Выполните `REFRESH MATERIALIZED VIEW CONCURRENTLY` и убедитесь, что данные обновились.

In [43]:
# Упражнение 5. VIEW с WITH CHECK OPTION

# Что делаем: создаём представление, которое показывает только рейсы из SVO. 
# WITH CHECK OPTION не даёт через это представление добавить рейс из другого аэропорта.

# представление только с рейсами из SVO
q("""
CREATE OR REPLACE VIEW v_svo_flights AS
SELECT flight_id, flight_no, departure_airport, scheduled_departure
FROM flights
WHERE departure_airport = 'SVO'
WITH CHECK OPTION      -- строки не из SVO через это VIEW вставить нельзя
""")

# 1. корректная вставка (SVO), должна пройти
q("""
INSERT INTO v_svo_flights (flight_no, departure_airport, scheduled_departure)
VALUES ('XX111', 'SVO', '2025-06-16 10:00:00')
""")
print("SVO вставлен")

# 2. вставка с DME, должна быть отклонена
try:
    q("""
    INSERT INTO v_svo_flights (flight_no, departure_airport, scheduled_departure)
    VALUES ('XX222', 'DME', '2025-06-16 11:00:00')
    """)
except psycopg.errors.WithCheckOptionViolation as e:   # правильный тип ошибки
    print("DME отклонён →", str(e).splitlines()[0])

# 3. в представлении только SVO-рейсы
print("Рейсов в v_svo_flights:", q("SELECT count(*) FROM v_svo_flights")[0][0])

SVO вставлен
DME отклонён → new row violates check option for view "v_svo_flights"
Рейсов в v_svo_flights: 3


In [44]:
# Упражнение 6. MATERIALIZED VIEW и REFRESH CONCURRENTLY

# Что делаем: сохраняем «снимок» с количеством рейсов по аэропортам.
# Добавляем рейс и видим, что снимок не изменился сам.
# Затем обновляем его вручную.
# Для CONCURRENTLY обязателен уникальный индекс.

q("DROP MATERIALIZED VIEW IF EXISTS mv_airport_stats")

# сохраняем результат запроса как физическую таблицу-снимок
q("""
CREATE MATERIALIZED VIEW mv_airport_stats AS
SELECT departure_airport, count(*) AS flights_cnt
FROM flights
GROUP BY departure_airport
""")

# уникальный индекс нужен, чтобы работал REFRESH ... CONCURRENTLY
q("CREATE UNIQUE INDEX idx_mv_airport_stats ON mv_airport_stats(departure_airport)")

# 2. смотрим снимок сейчас
print("Сначала:", q("SELECT * FROM mv_airport_stats ORDER BY departure_airport"))

# 3. добавляем новый рейс из нового аэропорта LED
q("""
INSERT INTO flights (flight_no, departure_airport, scheduled_departure)
VALUES ('NEW01', 'LED', '2025-06-17 09:00:00')
""")

# 4. снимок не изменился: LED в нём нет
print("После вставки:", q("SELECT * FROM mv_airport_stats ORDER BY departure_airport"))

# 5. обновляем снимок без блокировки чтения
q("REFRESH MATERIALIZED VIEW CONCURRENTLY mv_airport_stats")

# теперь LED появился
print("После REFRESH:", q("SELECT * FROM mv_airport_stats ORDER BY departure_airport"))

Сначала: [('DME', 3), ('SVO', 3)]
После вставки: [('DME', 3), ('SVO', 3)]
После REFRESH: [('DME', 3), ('LED', 1), ('SVO', 3)]
